# Part 2 — Group Presentation & Notebook

**Group members:** 

- Dargys Perez : `Data Engineer`
  
- Jessika Rosen : `Data Analyst`
  
- Cecilia Back : `ERP Engineer`

Show data evidence during the presentation — in your Notebook or PowerPoint using Python as your data processing language

In [2]:
# using pandas for analysis
import pandas as pd

In [3]:
# load the relevant data for the 10 rules analysis
invoice_line = pd.read_csv("invoice_line.csv")
deliveries = pd.read_csv("deliveries.csv")
customers = pd.read_csv("customers.csv")

In [8]:
deliveries.head()

,delivery_confirm_id,order_id,delivery_confirm_date,delivery_country
0,CONF-747653,ORD-2027-296460,2027-02-10,France
1,CONF-650282,ORD-2026-865441,2027-01-13,France
2,CONF-997937,ORD-2027-449701,2027-03-03,Italy
3,CONF-101069,ORD-2027-361984,2027-03-19T11:33:00,Germany
4,CONF-406108,ORD-2027-278275,19/03/2027,Netherlands


In [5]:
invoice_line.tail()

,invoice_line_id,invoice_id,invoice_date,invoice_type,order_id,product_id,product_description,quantity,uom,unit_price,currency,line_amount,vat_rate,vat_amount,total_amount,invoice_status,invoice_ref,vat_category_code
98,IL-001,INV-2027-642637,2027-03-10,invoice,ORD-2027-850022,PROD-192837,Droxel 65W GaN Charger,86,pcs,28.4,EUR,2442.4,0.0,0.00,2442.40,issued,NaN,AE
99,IL-003,INV-2027-267919,2027-03-09,invoice,ORD-2027-449701,PROD-461829,Plonex Webcam 1080p Pro,36,pcs,44.0,EUR,1584.0,0.0,0.00,1584.00,issued,NaN,AE
100,IL-001,INV-2027-990042,2027-02-14,invoice,ORD-2026-543670,PROD-192837,Droxel 65W GaN Charger,48,pcs,28.4,EUR,1363.2,19.0,259.01,1622.21,issued,NaN,S
101,IL-001,INV-2027-140410,2027-03-05,invoice,ORD-2027-901318,PROD-639182,Blipcore Wireless Adapter AC1200,42,pcs,22.8,EUR,957.6,0.0,0.00,957.60,issued,NaN,AE
102,IL-001,INV-2027-226551,2027-03-09,invoice,ORD-2027-152306,PROD-738291,Quorvex 2TB NVMe SSD M.2,83,pcs,112.5,EUR,9337.5,0.0,0.00,9337.50,cancelled,NaN,AE


Address at least one concrete data quality issue found in the datasets

In [14]:
#Join datasets
df = invoice_line.merge(
    deliveries,
    on="order_id",
    how="left"
)

In [15]:
#Convert to datetime
df["invoice_date"] = pd.to_datetime(df["invoice_date"])


In [16]:
# the delivery_confirm_date has data quality issues, the date is given in differents formats and I need first to normalize.
df["delivery_confirm_date"] = pd.to_datetime(
    df["delivery_confirm_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)


In [17]:
# I wanted to check if the datatypes where compatible with each other.
df.dtypes

invoice_line_id                     str
invoice_id                          str
invoice_date             datetime64[us]
invoice_type                        str
order_id                            str
product_id                          str
product_description                 str
quantity                          int64
uom                                 str
unit_price                      float64
currency                            str
line_amount                     float64
vat_rate                        float64
vat_amount                      float64
total_amount                    float64
invoice_status                      str
invoice_ref                         str
vat_category_code                   str
delivery_confirm_id                 str
delivery_confirm_date    datetime64[us]
delivery_country                    str
dtype: object

>Note: As we see the invoice_date and  delivery_confirm_date are both in the same format, ready for the next step

In [18]:

df["days_diff"] = (df["invoice_date"] - df["delivery_confirm_date"]).dt.days

In [19]:
# Here are we "Grouping By: count()" as we know from SQL, it counts the invoices that where send late and the ones that where on time
df["vida_10_day_rule"] = df["days_diff"].apply(
    lambda x: "Compliant" if x <= 10 else "Late"
)

In [20]:
#Just showing the results
df["vida_10_day_rule"].value_counts()

vida_10_day_rule
Compliant    71
Late         32
Name: count, dtype: int64